# Experiment: Stochastic Chart Transport Gauge Patch

Objective:
- Implement the stochastic gauge-patch version of chart transport as a standalone notebook.
- Keep the chart, critic, and artifact semantics explicit in notebook cells instead of delegating the method to the deterministic runtime.

This notebook patches the base study in three ways:
1. The encoder is stochastic: `y = f_phi(x, eta)` with `eta ~ N(0, I)` and `gauge_dimension = latent_dimension`.
2. The critic is side-conditioned and score-matches both data-side and model-side stochastic latent pushforwards.
3. Integrated training repairs reconstruction, refreshes the critic, and transports both latent pushforwards toward the symmetry-broken prior.

The default profile is the full ambient-128 setup: `2000` chart-pretrain steps, `2000` critic-pretrain steps, and `100_000` integrated steps, writing artifacts under `artifacts/multimodal_gaussian/<run_name>`.
Switch `profile_name` below to the alternate presets when you want a shorter smoke run or a different study regime.


In [1]:
from __future__ import annotations

import contextlib
import hashlib
import json
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import plotly.graph_objects as go
import polars as pl
import torch
import wandb
import torch.nn as nn
import torch.nn.functional as F
from plotly.subplots import make_subplots
from torch.nn.utils import clip_grad_norm_
from tqdm.auto import tqdm

from src.chart_transport.field import UniformVelocityMatchingSchedule
from src.config.base import BaseConfig
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import TimeConditioningConfig
from src.monitoring.utils import (
    MonitorStage,
    build_latent_grid,
    latent_square_limits,
    marker_color,
    normalize_vectors,
    project_latents_to_pca_space,
    project_latent_vectors_to_pca_space,
    sample_mode_batch,
    step_folder,
    vector_display_length,
    write_figure,
    write_mode_value_parquet,
)
from src.priors.anchored import AnchoredGaussianScaleMixturePriorConfig


## Profiles

The default ambient-128 preset is now the full setup: `2000` chart steps, `2000` critic steps, and `100_000` integrated steps.
Integrated artifacts are saved every `50` steps during the default run, with the final step included explicitly.
The other presets remain available when you want a shorter smoke run or a different study regime.


In [2]:
class DataSpecConfig(BaseConfig):
    num_modes: int
    ambient_dimension: int
    mode_std: float
    offset: float


class PriorSpecConfig(BaseConfig):
    precision: float


class ArchitectureConfig(BaseConfig):
    latent_dimension: int
    gauge_dimension: int
    hidden_dimension: int
    n_hidden_layers: int
    time_embedding_dimension: int


class OptimizationConfig(BaseConfig):
    chart_lr: float
    critic_lr: float
    grad_clip_norm: float
    latent_norm_weight: float
    stochastic_latent_l2_weight: float
    stochastic_latent_mean_weight: float
    stochastic_latent_batch_mean_weight: float
    stochastic_latent_variance_weight: float
    model_anchor_divisor: float
    latent_norm_delta: float
    reconstruction_delta: float


class TransportSpecConfig(BaseConfig):
    transport_step_size: float
    transport_step_cap: float
    num_time_samples: int
    t_range: tuple[float, float]
    critic_updates_per_transport_step: int


class ScheduleConfig(BaseConfig):
    train_batch_size: int
    pretrain_steps: int
    critic_steps: int
    integrated_steps: int
    pretrain_log_every: int
    critic_log_every: int
    integrated_log_every: int


class MonitoringConfig(BaseConfig):
    run_pretrain_monitor: bool
    run_critic_monitor: bool
    integrated_monitor_steps: list[int]
    eval_samples_per_mode: int
    generated_samples: int
    kde_scales: list[float]
    kl_num_samples: int
    avg_kl_num_batches: int
    field_grid_resolution: int


class GaugePatchNotebookConfig(BaseConfig):
    profile_name: str
    notes: str
    seed: int
    cuda_device: int
    wandb_project: str
    reuse_pretrain_cache: bool
    artifact_root: Path
    run_name: str
    data: DataSpecConfig
    prior: PriorSpecConfig
    architecture: ArchitectureConfig
    optimization: OptimizationConfig
    transport: TransportSpecConfig
    schedule: ScheduleConfig
    monitoring: MonitoringConfig


def build_experiment_config(profile_name: str) -> GaugePatchNotebookConfig:
    if profile_name not in {
        'full_ambient128',
        'smoke_ambient128',
        'study_ambient128',
        'study_2mode_baseline',
    }:
        raise ValueError(f'Unknown profile_name: {profile_name}')

    architecture = ArchitectureConfig(
        latent_dimension=128,
        gauge_dimension=128,
        hidden_dimension=512,
        n_hidden_layers=6,
        time_embedding_dimension=512,
    )
    optimization = OptimizationConfig(
        chart_lr=3e-4,
        critic_lr=3e-4,
        grad_clip_norm=1.0,
        latent_norm_weight=1e-4,
        stochastic_latent_l2_weight=1e-2,
        stochastic_latent_mean_weight=5e-3,
        stochastic_latent_batch_mean_weight=1.0,
        stochastic_latent_variance_weight=1.0,
        model_anchor_divisor=5.0,
        latent_norm_delta=5.0,
        reconstruction_delta=5.0,
    )
    transport = TransportSpecConfig(
        transport_step_size=0.05,
        transport_step_cap=0.05,
        num_time_samples=8,
        t_range=(0.01, 0.99),
        critic_updates_per_transport_step=4,
    )

    if profile_name == 'full_ambient128':
        data = DataSpecConfig(
            num_modes=8,
            ambient_dimension=128,
            mode_std=0.1,
            offset=0.0,
        )
        schedule = ScheduleConfig(
            train_batch_size=4096,
            pretrain_steps=2000,
            critic_steps=2000,
            integrated_steps=100_000,
            pretrain_log_every=100,
            critic_log_every=100,
            integrated_log_every=100,
        )
        monitoring = MonitoringConfig(
            run_pretrain_monitor=True,
            run_critic_monitor=False,
            integrated_monitor_steps=[*range(50, 100001, 50), 100_000],
            eval_samples_per_mode=512,
            generated_samples=4096,
            kde_scales=[1e-3, 3e-3, 1e-2, 3e-2],
            kl_num_samples=512,
            avg_kl_num_batches=8,
            field_grid_resolution=31,
        )
        notes = (
            'Full ambient-128 setup. Uses the 8-mode width-512 depth-6 '
            'architecture with train batch 4096, eval batch 4096, 2k chart steps, '
            '2k critic steps, integrated logging every 100 steps, 100k integrated '
            'steps, model_anchor_divisor=5 for the model-side pretrain anchor, '
            'and reusable chart/critic warm-start checkpoints.'
        )
    elif profile_name == 'smoke_ambient128':
        data = DataSpecConfig(
            num_modes=8,
            ambient_dimension=128,
            mode_std=0.1,
            offset=0.0,
        )
        schedule = ScheduleConfig(
            train_batch_size=192,
            pretrain_steps=25,
            critic_steps=25,
            integrated_steps=50,
            pretrain_log_every=5,
            critic_log_every=5,
            integrated_log_every=10,
        )
        monitoring = MonitoringConfig(
            run_pretrain_monitor=True,
            run_critic_monitor=False,
            integrated_monitor_steps=[50],
            eval_samples_per_mode=128,
            generated_samples=1024,
            kde_scales=[1e-3, 3e-3, 1e-2, 3e-2],
            kl_num_samples=128,
            avg_kl_num_batches=2,
            field_grid_resolution=15,
        )
        notes = (
            'Smoke profile for notebook execution. Uses the 8-mode width-512 depth-6 '
            'architecture from the scalability study, but with short schedules.'
        )
    elif profile_name == 'study_ambient128':
        data = DataSpecConfig(
            num_modes=8,
            ambient_dimension=128,
            mode_std=0.1,
            offset=0.0,
        )
        schedule = ScheduleConfig(
            train_batch_size=8192,
            pretrain_steps=1500,
            critic_steps=2000,
            integrated_steps=24000,
            pretrain_log_every=100,
            critic_log_every=100,
            integrated_log_every=1000,
        )
        monitoring = MonitoringConfig(
            run_pretrain_monitor=True,
            run_critic_monitor=True,
            integrated_monitor_steps=list(range(1, 24_000 + 1)),
            eval_samples_per_mode=500,
            generated_samples=3000,
            kde_scales=[1e-3, 3e-3, 1e-2, 3e-2],
            kl_num_samples=512,
            avg_kl_num_batches=8,
            field_grid_resolution=31,
        )
        notes = (
            'Study profile matching the stable 8-mode capacity-scaling regime: '
            'batch 192, width 512, depth 6, 24k integrated steps.'
        )
    else:
        data = DataSpecConfig(
            num_modes=2,
            ambient_dimension=2,
            mode_std=0.1,
            offset=0.0,
        )
        schedule = ScheduleConfig(
            train_batch_size=4096,
            pretrain_steps=5000,
            critic_steps=5000,
            integrated_steps=18000,
            pretrain_log_every=100,
            critic_log_every=100,
            integrated_log_every=1000,
        )
        monitoring = MonitoringConfig(
            run_pretrain_monitor=True,
            run_critic_monitor=False,
            integrated_monitor_steps=[*range(50, 18001, 50), 18_000],
            eval_samples_per_mode=500,
            generated_samples=3000,
            kde_scales=[1e-3, 3e-3, 1e-2, 3e-2],
            kl_num_samples=512,
            avg_kl_num_batches=8,
            field_grid_resolution=31,
        )
        notes = (
            'Study profile matching the 2-mode robust baseline: batch 256 and 18k '
            'integrated steps.'
        )

    return GaugePatchNotebookConfig(
        profile_name=profile_name,
        notes=notes,
        seed=0,
        cuda_device=1,
        wandb_project='diffusive-latent-generation',
        reuse_pretrain_cache=True,
        artifact_root=Path('/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian'),
        run_name=f"{datetime.now():%m%d%H%M%S}-{profile_name}-{uuid4().hex[:6]}",
        data=data,
        prior=PriorSpecConfig(precision=4.0),
        architecture=architecture,
        optimization=optimization,
        transport=transport,
        schedule=schedule,
        monitoring=monitoring,
    )


In [3]:
profile_name = 'study_ambient128'
experiment_config = build_experiment_config(profile_name=profile_name)
experiment_config.visualize()


## Instantiate Data, Prior, And Artifact Folder

The config above is intentionally lightweight so `.visualize()` stays readable.
This cell materializes the concrete `MultimodalGaussianDataConfig` and symmetry-broken prior used by the actual experiment.


In [4]:
torch.manual_seed(experiment_config.seed)
torch.cuda.manual_seed_all(experiment_config.seed)
torch.set_float32_matmul_precision('medium')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this notebook')
if experiment_config.cuda_device >= torch.cuda.device_count():
    raise ValueError(
        f"cuda_device={experiment_config.cuda_device} but only {torch.cuda.device_count()} devices are available"
    )

device = torch.device(f'cuda:{experiment_config.cuda_device}')
run_folder = experiment_config.artifact_root / experiment_config.run_name
run_folder.mkdir(parents=True, exist_ok=False)

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=experiment_config.data.num_modes,
    mode_std=experiment_config.data.mode_std,
    offset=experiment_config.data.offset,
    ambient_dimension=experiment_config.data.ambient_dimension,
).to(device=device)
prior_config = AnchoredGaussianScaleMixturePriorConfig.initialize(
    latent_shape=[experiment_config.architecture.latent_dimension],
    precision=experiment_config.prior.precision,
)
transport_schedule = UniformVelocityMatchingSchedule()

wandb_run = wandb.init(
    project=experiment_config.wandb_project,
    name=experiment_config.run_name,
    dir=str(run_folder),
    config=experiment_config.model_dump(mode='json'),
)
wandb.summary['run_folder'] = str(run_folder)

{
    'device': str(device),
    'run_folder': str(run_folder),
    'num_modes': data_config.num_modes,
    'ambient_dimension': data_config.ambient_dimension,
    'latent_dimension': experiment_config.architecture.latent_dimension,
}


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nlyu/.netrc.


wandb: Currently logged in as: lyuxingjian (lyuxingjian-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Serializing object of type list that is 192056 bytes


wandb: WARNING Serializing object of type list that is 194680 bytes


{'device': 'cuda:1',
 'run_folder': '/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian/0401011438-study_ambient128-cdd910',
 'num_modes': 8,
 'ambient_dimension': 128,
 'latent_dimension': 128}

## Models And Optimizers

The encoder concatenates `x` and the gauge sample `eta`.
The critic concatenates the latent input with a two-class side embedding for `data` vs `model` and is time-conditioned using the shared residual MLP blocks.


In [5]:
def make_residual_mlp(*, input_dim: int, output_dim: int, condition_time: bool = False) -> nn.Module:
    time_config = None
    if condition_time:
        time_config = TimeConditioningConfig(
            min_t_lambda=experiment_config.transport.t_range[0],
            max_t_lambda=experiment_config.transport.t_range[1],
            sinusoidal_dim=experiment_config.architecture.time_embedding_dimension,
            hidden_dim=experiment_config.architecture.hidden_dimension,
            output_dim=experiment_config.architecture.time_embedding_dimension,
        )
    layer_dims = [
        input_dim,
        *([experiment_config.architecture.hidden_dimension] * experiment_config.architecture.n_hidden_layers),
        output_dim,
    ]
    return StackedResidualMLPConfig.initialize(
        layer_dims=layer_dims,
        time_conditioning_config=time_config,
    ).get_model().to(device=device)


class GaugeEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.backbone = make_residual_mlp(
            input_dim=data_config.data_numel() + experiment_config.architecture.gauge_dimension,
            output_dim=experiment_config.architecture.latent_dimension,
        )

    def forward(self, x: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
        return self.backbone(torch.cat([x, eta], dim=-1))


class GaugeDecoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.backbone = make_residual_mlp(
            input_dim=experiment_config.architecture.latent_dimension,
            output_dim=data_config.data_numel(),
        )

    def forward(self, y: torch.Tensor) -> torch.Tensor:
        return self.backbone(y)


class SideConditionedCritic(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.backbone = make_residual_mlp(
            input_dim=experiment_config.architecture.latent_dimension + 2,
            output_dim=experiment_config.architecture.latent_dimension,
            condition_time=True,
        )

    def forward(self, y_t: torch.Tensor, t: torch.Tensor, side: torch.Tensor) -> torch.Tensor:
        side_one_hot = F.one_hot(side.to(dtype=torch.long), num_classes=2).to(dtype=y_t.dtype)
        return self.backbone(torch.cat([y_t, side_one_hot], dim=-1), t)


def build_pretrain_cache_config(config: GaugePatchNotebookConfig) -> dict[str, object]:
    return {
        'cache_version': 1,
        'seed': config.seed,
        'data': config.data.model_dump(mode='json'),
        'prior': config.prior.model_dump(mode='json'),
        'architecture': config.architecture.model_dump(mode='json'),
        'optimization': config.optimization.model_dump(mode='json'),
        'critic_transport': {
            't_range': list(config.transport.t_range),
        },
        'schedule': {
            'train_batch_size': config.schedule.train_batch_size,
            'pretrain_steps': config.schedule.pretrain_steps,
            'critic_steps': config.schedule.critic_steps,
        },
    }


def build_pretrain_cache_key(config: GaugePatchNotebookConfig) -> str:
    payload = json.dumps(build_pretrain_cache_config(config), sort_keys=True)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()[:16]


@dataclass(kw_only=True)
class NotebookRuntime:
    config: GaugePatchNotebookConfig
    data_config: MultimodalGaussianDataConfig
    prior_config: AnchoredGaussianScaleMixturePriorConfig
    transport_schedule: UniformVelocityMatchingSchedule
    device: torch.device
    run_folder: Path
    encoder: GaugeEncoder
    decoder: GaugeDecoder
    critic: SideConditionedCritic
    chart_optimizer: torch.optim.Optimizer
    critic_optimizer: torch.optim.Optimizer
    pretrain_cache_key: str
    pretrain_cache_path: Path
    reuse_pretrain_cache: bool
    pretrain_cache_hit: bool = False
    pretrain_cache_metadata: dict[str, object] | None = None


def parameter_count(module: nn.Module) -> int:
    return sum(parameter.numel() for parameter in module.parameters())


encoder = GaugeEncoder()
decoder = GaugeDecoder()
critic = SideConditionedCritic()
chart_optimizer = torch.optim.AdamW(
    [
        {'params': encoder.parameters(), 'lr': experiment_config.optimization.chart_lr},
        {'params': decoder.parameters(), 'lr': experiment_config.optimization.chart_lr},
    ]
)
critic_optimizer = torch.optim.AdamW(
    critic.parameters(),
    lr=experiment_config.optimization.critic_lr,
)

pretrain_cache_key = build_pretrain_cache_key(experiment_config)
pretrain_cache_path = experiment_config.artifact_root / '_pretrain_cache' / f'{pretrain_cache_key}.pt'

runtime = NotebookRuntime(
    config=experiment_config,
    data_config=data_config,
    prior_config=prior_config,
    transport_schedule=transport_schedule,
    device=device,
    run_folder=run_folder,
    encoder=encoder,
    decoder=decoder,
    critic=critic,
    chart_optimizer=chart_optimizer,
    critic_optimizer=critic_optimizer,
    pretrain_cache_key=pretrain_cache_key,
    pretrain_cache_path=pretrain_cache_path,
    reuse_pretrain_cache=experiment_config.reuse_pretrain_cache,
)

wandb.summary['pretrain_cache_key'] = runtime.pretrain_cache_key
wandb.summary['pretrain_cache_path'] = str(runtime.pretrain_cache_path)
wandb.summary['reuse_pretrain_cache'] = runtime.reuse_pretrain_cache

{
    'encoder_parameters': parameter_count(runtime.encoder),
    'decoder_parameters': parameter_count(runtime.decoder),
    'critic_parameters': parameter_count(runtime.critic),
    'pretrain_cache_key': runtime.pretrain_cache_key,
    'pretrain_cache_path': str(runtime.pretrain_cache_path),
}


{'encoder_parameters': 6898944,
 'decoder_parameters': 6702080,
 'critic_parameters': 8873094,
 'pretrain_cache_key': '8c65d38bb6c9b43b',
 'pretrain_cache_path': '/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian/_pretrain_cache/8c65d38bb6c9b43b.pt'}

## Sampling And Training Semantics

`sample_data_latents` draws `x ~ rho_data`, `eta ~ N(0, I)`, and returns `f_phi(x, eta)`.
`sample_model_latents` draws `z ~ prior`, decodes to `g_theta(z)`, draws a fresh gauge sample, and re-encodes to the stochastic model latent `f_phi(g_theta(z), eta)`.

Transport is estimated separately for each side but with the same score-matching machinery and the same symmetry-broken prior score.

By default the latent ambient dimension is `128`, so `data_eta_shape` and `model_eta_shape` should both have trailing dimension `128`. The latent plots below are therefore 2D PCA projections of a 128D stochastic latent space, and the default run budget is `2000 / 2000 / 100_000` for pretrain, critic, and integrated phases.


In [6]:
DATA_SIDE = 0
MODEL_SIDE = 1


@contextlib.contextmanager
def precision_context(runtime: NotebookRuntime):
    with torch.cuda.device(runtime.device):
        with torch.device(str(runtime.device)):
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                yield


def sample_gauge(runtime: NotebookRuntime, batch_size: int) -> torch.Tensor:
    return torch.randn(
        batch_size,
        runtime.config.architecture.gauge_dimension,
        device=runtime.device,
        dtype=torch.float32,
    )


def zero_gauge(runtime: NotebookRuntime, batch_size: int) -> torch.Tensor:
    return torch.zeros(
        batch_size,
        runtime.config.architecture.gauge_dimension,
        device=runtime.device,
        dtype=torch.float32,
    )


def sample_data_latents(
    runtime: NotebookRuntime,
    *,
    batch_size: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    data_batch = runtime.data_config.sample_unconditional(batch_size=batch_size).to(dtype=torch.float32)
    eta = sample_gauge(runtime, batch_size)
    latents = runtime.encoder(data_batch, eta)
    return data_batch, eta, latents


def sample_model_latents(
    runtime: NotebookRuntime,
    *,
    batch_size: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    prior_latents = runtime.prior_config.sample(batch_size=batch_size).to(
        device=runtime.device,
        dtype=torch.float32,
    )
    model_samples = runtime.decoder(prior_latents)
    eta = sample_gauge(runtime, batch_size)
    model_latents = runtime.encoder(model_samples, eta)
    return prior_latents, model_samples, eta, model_latents


def assign_mode_ids(runtime: NotebookRuntime, samples: torch.Tensor) -> torch.Tensor:
    projected = runtime.data_config.project(samples.float())
    centers = runtime.data_config.mode_centers_2d().to(projected)
    return torch.cdist(projected, centers).argmin(dim=-1)


def latent_anchor_loss(runtime: NotebookRuntime, latents: torch.Tensor) -> torch.Tensor:
    latent_norms = latents.float().norm(dim=-1)
    return F.huber_loss(
        latent_norms,
        torch.zeros_like(latent_norms),
        delta=runtime.config.optimization.latent_norm_delta,
        reduction='mean',
    )


def stochastic_latent_l2_loss(latents: torch.Tensor) -> torch.Tensor:
    return latents.float().square().mean()


def stochastic_latent_mean_loss(latents: torch.Tensor) -> torch.Tensor:
    per_sample_mean = latents.float().mean(dim=-1)
    return F.mse_loss(per_sample_mean, torch.zeros_like(per_sample_mean))


def stochastic_latent_batch_mean_loss(latents: torch.Tensor) -> torch.Tensor:
    per_dimension_batch_mean = latents.float().mean(dim=0)
    return F.mse_loss(per_dimension_batch_mean, torch.zeros_like(per_dimension_batch_mean))


def stochastic_latent_variance_loss(latents: torch.Tensor) -> torch.Tensor:
    centered_latents = latents.float() - latents.float().mean(dim=0, keepdim=True)
    per_dimension_variance = centered_latents.square().mean(dim=0)
    return F.mse_loss(per_dimension_variance, torch.ones_like(per_dimension_variance))


def sample_transport_times(
    runtime: NotebookRuntime,
    *,
    batch_size: int,
    num_time_samples: int | None = None,
) -> torch.Tensor:
    t_min, t_max = runtime.config.transport.t_range
    if num_time_samples is None:
        return t_min + (t_max - t_min) * torch.rand(batch_size, device=runtime.device)
    bin_edges = torch.linspace(t_min, t_max, num_time_samples + 1, device=runtime.device)
    bin_starts = bin_edges[:-1]
    bin_widths = bin_edges[1:] - bin_edges[:-1]
    return bin_starts.unsqueeze(0) + bin_widths.unsqueeze(0) * torch.rand(
        batch_size,
        num_time_samples,
        device=runtime.device,
    )


def critic_loss_for_side(
    runtime: NotebookRuntime,
    *,
    clean_latents: torch.Tensor,
    side_value: int,
) -> torch.Tensor:
    t = sample_transport_times(runtime, batch_size=clean_latents.shape[0])
    eps = torch.randn_like(clean_latents)
    noised_latents = (1.0 - t).unsqueeze(-1) * clean_latents + t.unsqueeze(-1) * eps
    side = torch.full((clean_latents.shape[0],), side_value, device=runtime.device, dtype=torch.long)
    predicted_noise = runtime.critic(noised_latents, t, side)
    return F.mse_loss(predicted_noise.float(), eps.float())


def estimate_weighted_score_components(
    runtime: NotebookRuntime,
    *,
    clean_latents: torch.Tensor,
    side_value: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    num_time_samples = runtime.config.transport.num_time_samples
    transport_t = sample_transport_times(
        runtime,
        batch_size=clean_latents.shape[0],
        num_time_samples=num_time_samples,
    )
    source_latents = clean_latents.detach().unsqueeze(1).expand(-1, num_time_samples, -1)
    transport_eps = torch.randn_like(source_latents)

    transport_t = torch.cat([transport_t, transport_t], dim=1)
    source_latents = torch.cat([source_latents, source_latents], dim=1)
    transport_eps = torch.cat([transport_eps, -transport_eps], dim=1)

    noised_latents = (
        (1.0 - transport_t).unsqueeze(-1) * source_latents
        + transport_t.unsqueeze(-1) * transport_eps
    )
    flat_noised_latents = noised_latents.reshape(-1, runtime.config.architecture.latent_dimension)
    flat_t = transport_t.reshape(-1)
    flat_side = torch.full((flat_t.shape[0],), side_value, device=runtime.device, dtype=torch.long)

    predicted_noise = runtime.critic(flat_noised_latents, flat_t, flat_side).reshape_as(noised_latents)
    score_estimate = -(predicted_noise / transport_t.unsqueeze(-1))
    prior_score = runtime.prior_config.analytic_score(
        flat_noised_latents.float(),
        flat_t.float(),
    ).to(dtype=noised_latents.dtype).reshape_as(noised_latents)
    pullback_weight = runtime.transport_schedule.pullback_weight(flat_t.float()).to(
        dtype=noised_latents.dtype
    ).reshape_as(transport_t)
    weighted_score_terms = pullback_weight.unsqueeze(-1) * score_estimate
    weighted_prior_terms = pullback_weight.unsqueeze(-1) * prior_score

    midpoint = num_time_samples
    weighted_score_terms = 0.5 * (
        weighted_score_terms[:, :midpoint] + weighted_score_terms[:, midpoint:]
    )
    weighted_prior_terms = 0.5 * (
        weighted_prior_terms[:, :midpoint] + weighted_prior_terms[:, midpoint:]
    )
    weighted_score = weighted_score_terms.mean(dim=1)
    weighted_prior = weighted_prior_terms.mean(dim=1)
    field = weighted_prior - weighted_score
    return weighted_score, weighted_prior, field


def estimate_transport_targets(
    runtime: NotebookRuntime,
    *,
    clean_latents: torch.Tensor,
    side_value: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    _, _, field = estimate_weighted_score_components(
        runtime,
        clean_latents=clean_latents,
        side_value=side_value,
    )
    field_norm = field.float().norm(dim=-1, keepdim=True)
    step_size = torch.minimum(
        torch.full_like(field_norm, runtime.config.transport.transport_step_size),
        torch.full_like(field_norm, runtime.config.transport.transport_step_cap)
        / field_norm.clamp_min(1e-6),
    )
    transport_step = step_size * field
    return clean_latents.detach() + transport_step, field, field_norm


def pretrain_step(runtime: NotebookRuntime) -> dict[str, float]:
    runtime.encoder.train()
    runtime.decoder.train()
    runtime.critic.eval()
    runtime.chart_optimizer.zero_grad(set_to_none=True)

    batch_size = runtime.config.schedule.train_batch_size
    with precision_context(runtime):
        data_batch, _, data_latents = sample_data_latents(
            runtime,
            batch_size=batch_size,
        )
        data_fiber_reconstructions = runtime.decoder(data_latents)
        data_fiber_reconstruction_loss = F.huber_loss(
            data_fiber_reconstructions,
            data_batch,
            delta=runtime.config.optimization.reconstruction_delta,
            reduction='mean',
        )

        canonical_eta = zero_gauge(runtime, batch_size)
        data_canonical_latents = runtime.encoder(data_batch, canonical_eta)
        data_canonical_reconstructions = runtime.decoder(data_canonical_latents)
        data_canonical_reconstruction_loss = F.huber_loss(
            data_canonical_reconstructions,
            data_batch,
            delta=runtime.config.optimization.reconstruction_delta,
            reduction='mean',
        )

        prior_latents = runtime.prior_config.sample(batch_size=batch_size).to(
            device=runtime.device,
            dtype=torch.float32,
        )
        with torch.no_grad():
            canonical_model_samples = runtime.decoder(prior_latents)
            canonical_model_latents = runtime.encoder(canonical_model_samples, canonical_eta)
        canonical_model_reconstructions = runtime.decoder(canonical_model_latents)
        canonical_model_roundtrip_latents = runtime.encoder(
            canonical_model_reconstructions,
            canonical_eta,
        )
        canonical_model_decoder_loss = F.huber_loss(
            canonical_model_reconstructions,
            canonical_model_samples,
            delta=runtime.config.optimization.reconstruction_delta,
            reduction='mean',
        )
        canonical_model_encoder_loss = F.mse_loss(
            canonical_model_roundtrip_latents,
            canonical_model_latents,
        )

        _, model_samples, _, model_latents = sample_model_latents(
            runtime,
            batch_size=batch_size,
        )
        data_anchor = latent_anchor_loss(runtime, data_latents)
        model_anchor = latent_anchor_loss(runtime, model_latents)
        data_stochastic_l2 = stochastic_latent_l2_loss(data_latents)
        model_stochastic_l2 = stochastic_latent_l2_loss(model_latents)
        data_stochastic_mean = stochastic_latent_mean_loss(data_latents)
        model_stochastic_mean = stochastic_latent_mean_loss(model_latents)
        data_stochastic_batch_mean = stochastic_latent_batch_mean_loss(data_latents)
        model_stochastic_batch_mean = stochastic_latent_batch_mean_loss(model_latents)
        data_variance = stochastic_latent_variance_loss(data_latents)
        model_variance = stochastic_latent_variance_loss(model_latents)
        repair_loss = (
            data_fiber_reconstruction_loss
            + data_canonical_reconstruction_loss
            + canonical_model_decoder_loss
            + canonical_model_encoder_loss
        )
        weak_anchor_loss = runtime.config.optimization.latent_norm_weight * (
            data_anchor
            + model_anchor / runtime.config.optimization.model_anchor_divisor
        )
        stochastic_prior_loss = runtime.config.optimization.stochastic_latent_l2_weight * (
            data_stochastic_l2 + model_stochastic_l2
        ) + runtime.config.optimization.stochastic_latent_mean_weight * (
            data_stochastic_mean + model_stochastic_mean
        ) + runtime.config.optimization.stochastic_latent_batch_mean_weight * (
            data_stochastic_batch_mean + model_stochastic_batch_mean
        )
        variance_loss = runtime.config.optimization.stochastic_latent_variance_weight * (
            data_variance + model_variance
        )
        loss = repair_loss + weak_anchor_loss + stochastic_prior_loss + variance_loss

    loss.backward()
    clip_grad_norm_(
        [*runtime.encoder.parameters(), *runtime.decoder.parameters()],
        runtime.config.optimization.grad_clip_norm,
    )
    runtime.chart_optimizer.step()
    return {
        'pretrain_loss': float(loss.detach().cpu()),
        'pretrain_repair_loss': float(repair_loss.detach().cpu()),
        'data_reconstruction_loss': float(data_fiber_reconstruction_loss.detach().cpu()),
        'data_canonical_reconstruction_loss': float(data_canonical_reconstruction_loss.detach().cpu()),
        'model_canonical_decoder_loss': float(canonical_model_decoder_loss.detach().cpu()),
        'model_canonical_encoder_loss': float(canonical_model_encoder_loss.detach().cpu()),
        'data_anchor_loss': float(data_anchor.detach().cpu()),
        'model_anchor_loss': float(model_anchor.detach().cpu()),
        'data_stochastic_l2_loss': float(data_stochastic_l2.detach().cpu()),
        'model_stochastic_l2_loss': float(model_stochastic_l2.detach().cpu()),
        'data_stochastic_mean_loss': float(data_stochastic_mean.detach().cpu()),
        'model_stochastic_mean_loss': float(model_stochastic_mean.detach().cpu()),
        'data_stochastic_batch_mean_loss': float(data_stochastic_batch_mean.detach().cpu()),
        'model_stochastic_batch_mean_loss': float(model_stochastic_batch_mean.detach().cpu()),
        'weak_anchor_loss': float(weak_anchor_loss.detach().cpu()),
        'stochastic_prior_loss': float(stochastic_prior_loss.detach().cpu()),
        'data_variance_loss': float(data_variance.detach().cpu()),
        'model_variance_loss': float(model_variance.detach().cpu()),
        'weighted_variance_loss': float(variance_loss.detach().cpu()),
        'weighted_model_anchor_loss': float(
            (model_anchor / runtime.config.optimization.model_anchor_divisor).detach().cpu()
        ),
        'model_sample_norm_mean': float(model_samples.detach().float().norm(dim=-1).mean().cpu()),
    }


def critic_step(runtime: NotebookRuntime) -> dict[str, float]:
    runtime.encoder.eval()
    runtime.decoder.eval()
    runtime.critic.train()
    runtime.critic_optimizer.zero_grad(set_to_none=True)

    with torch.no_grad():
        _, _, data_latents = sample_data_latents(
            runtime,
            batch_size=runtime.config.schedule.train_batch_size,
        )
        _, _, _, model_latents = sample_model_latents(
            runtime,
            batch_size=runtime.config.schedule.train_batch_size,
        )

    with precision_context(runtime):
        data_loss = critic_loss_for_side(runtime, clean_latents=data_latents, side_value=DATA_SIDE)
        model_loss = critic_loss_for_side(runtime, clean_latents=model_latents, side_value=MODEL_SIDE)
        loss = 0.5 * (data_loss + model_loss)

    loss.backward()
    clip_grad_norm_(runtime.critic.parameters(), runtime.config.optimization.grad_clip_norm)
    runtime.critic_optimizer.step()
    return {
        'critic_loss': float(loss.detach().cpu()),
        'critic_data_loss': float(data_loss.detach().cpu()),
        'critic_model_loss': float(model_loss.detach().cpu()),
    }


def repair_step(runtime: NotebookRuntime) -> dict[str, float]:
    runtime.encoder.train()
    runtime.decoder.train()
    runtime.critic.eval()
    runtime.chart_optimizer.zero_grad(set_to_none=True)

    batch_size = runtime.config.schedule.train_batch_size
    with precision_context(runtime):
        data_batch, _, data_latents = sample_data_latents(
            runtime,
            batch_size=batch_size,
        )
        data_fiber_reconstructions = runtime.decoder(data_latents)
        data_fiber_reconstruction_loss = F.huber_loss(
            data_fiber_reconstructions,
            data_batch,
            delta=runtime.config.optimization.reconstruction_delta,
            reduction='mean',
        )

        canonical_eta = zero_gauge(runtime, batch_size)
        data_canonical_latents = runtime.encoder(data_batch, canonical_eta)
        data_canonical_reconstructions = runtime.decoder(data_canonical_latents)
        data_canonical_reconstruction_loss = F.huber_loss(
            data_canonical_reconstructions,
            data_batch,
            delta=runtime.config.optimization.reconstruction_delta,
            reduction='mean',
        )

        prior_latents = runtime.prior_config.sample(batch_size=batch_size).to(
            device=runtime.device,
            dtype=torch.float32,
        )
        with torch.no_grad():
            canonical_model_samples = runtime.decoder(prior_latents)
            canonical_model_latents = runtime.encoder(canonical_model_samples, canonical_eta)
        canonical_model_reconstructions = runtime.decoder(canonical_model_latents)
        canonical_model_roundtrip_latents = runtime.encoder(
            canonical_model_reconstructions,
            canonical_eta,
        )
        canonical_model_decoder_loss = F.huber_loss(
            canonical_model_reconstructions,
            canonical_model_samples,
            delta=runtime.config.optimization.reconstruction_delta,
            reduction='mean',
        )
        canonical_model_encoder_loss = F.mse_loss(
            canonical_model_roundtrip_latents,
            canonical_model_latents,
        )
        loss = (
            data_fiber_reconstruction_loss
            + data_canonical_reconstruction_loss
            + canonical_model_decoder_loss
            + canonical_model_encoder_loss
        )

    loss.backward()
    clip_grad_norm_(
        [*runtime.encoder.parameters(), *runtime.decoder.parameters()],
        runtime.config.optimization.grad_clip_norm,
    )
    runtime.chart_optimizer.step()
    return {
        'repair_loss': float(loss.detach().cpu()),
        'repair_data_fiber_reconstruction_loss': float(data_fiber_reconstruction_loss.detach().cpu()),
        'repair_data_canonical_reconstruction_loss': float(data_canonical_reconstruction_loss.detach().cpu()),
        'repair_model_canonical_decoder_loss': float(canonical_model_decoder_loss.detach().cpu()),
        'repair_model_canonical_encoder_loss': float(canonical_model_encoder_loss.detach().cpu()),
    }


def transport_step(runtime: NotebookRuntime) -> dict[str, float]:
    runtime.encoder.train()
    runtime.decoder.train()
    runtime.critic.eval()
    runtime.chart_optimizer.zero_grad(set_to_none=True)

    batch_size = runtime.config.schedule.train_batch_size
    with precision_context(runtime):
        data_batch = runtime.data_config.sample_unconditional(batch_size=batch_size).to(dtype=torch.float32)
        data_eta = sample_gauge(runtime, batch_size)
        with torch.no_grad():
            frozen_data_latents = runtime.encoder(data_batch, data_eta)
            data_targets, _, data_field_norm = estimate_transport_targets(
                runtime,
                clean_latents=frozen_data_latents,
                side_value=DATA_SIDE,
            )
        data_latents = runtime.encoder(data_batch, data_eta)
        data_encoder_loss = F.mse_loss(data_latents, data_targets)
        data_decoder_loss = F.mse_loss(runtime.decoder(data_targets), data_batch.detach())

        prior_latents = runtime.prior_config.sample(batch_size=batch_size).to(
            device=runtime.device,
            dtype=torch.float32,
        )
        model_eta = sample_gauge(runtime, batch_size)
        with torch.no_grad():
            model_samples = runtime.decoder(prior_latents)
            frozen_model_latents = runtime.encoder(model_samples, model_eta)
            model_targets, _, model_field_norm = estimate_transport_targets(
                runtime,
                clean_latents=frozen_model_latents,
                side_value=MODEL_SIDE,
            )
        model_latents = runtime.encoder(model_samples.detach(), model_eta)
        model_encoder_loss = F.mse_loss(model_latents, model_targets)
        model_decoder_loss = F.mse_loss(runtime.decoder(model_targets), model_samples.detach())

        loss = (
            data_encoder_loss
            + data_decoder_loss
            + model_encoder_loss
            + model_decoder_loss
        )

    loss.backward()
    clip_grad_norm_(
        [*runtime.encoder.parameters(), *runtime.decoder.parameters()],
        runtime.config.optimization.grad_clip_norm,
    )
    runtime.chart_optimizer.step()
    return {
        'transport_loss': float(loss.detach().cpu()),
        'data_encoder_transport_loss': float(data_encoder_loss.detach().cpu()),
        'data_decoder_transport_loss': float(data_decoder_loss.detach().cpu()),
        'model_encoder_transport_loss': float(model_encoder_loss.detach().cpu()),
        'model_decoder_transport_loss': float(model_decoder_loss.detach().cpu()),
        'data_field_norm_mean': float(data_field_norm.mean().detach().cpu()),
        'model_field_norm_mean': float(model_field_norm.mean().detach().cpu()),
    }


def average_metrics(metrics_sequence: list[dict[str, float]]) -> dict[str, float]:
    keys = metrics_sequence[0].keys()
    return {
        key: sum(metrics[key] for metrics in metrics_sequence) / len(metrics_sequence)
        for key in keys
    }


def should_log(*, step: int, total_steps: int, every: int) -> bool:
    return step == 1 or step % every == 0 or step == total_steps


def log_stage_metrics_to_wandb(
    *,
    stage_name: str,
    step: int,
    metrics: dict[str, float],
) -> None:
    payload = {
        f'{stage_name}/step': step,
        **{f'{stage_name}/{key}': value for key, value in metrics.items()},
    }
    wandb.log(payload)


def log_monitor_snapshot_to_wandb(
    *,
    stage_name: str,
    step: int,
    metrics: dict[str, float],
    folder: Path,
) -> None:
    payload = {
        f'{stage_name}_monitor/step': step,
        f'{stage_name}_monitor/folder': str(folder),
        **{f'{stage_name}_monitor/{key}': value for key, value in metrics.items()},
    }
    wandb.log(payload)


def build_integrated_history_figure(
    integrated_history: list[dict[str, float]],
) -> go.Figure:
    if len(integrated_history) == 0:
        raise ValueError('integrated_history must be non-empty')

    steps = list(range(1, len(integrated_history) + 1))
    groups = [
        (
            'Repair',
            [
                'repair_loss',
                'repair_data_fiber_reconstruction_loss',
                'repair_data_canonical_reconstruction_loss',
                'repair_model_canonical_decoder_loss',
                'repair_model_canonical_encoder_loss',
            ],
        ),
        (
            'Critic',
            [
                'critic_loss',
                'critic_data_loss',
                'critic_model_loss',
            ],
        ),
        (
            'Transport',
            [
                'transport_loss',
                'data_encoder_transport_loss',
                'data_decoder_transport_loss',
                'model_encoder_transport_loss',
                'model_decoder_transport_loss',
            ],
        ),
        (
            'Field Norms',
            [
                'data_field_norm_mean',
                'model_field_norm_mean',
            ],
        ),
    ]

    figure = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[title for title, _ in groups],
        horizontal_spacing=0.12,
        vertical_spacing=0.15,
    )
    for panel_index, (panel_title, metric_names) in enumerate(groups, start=1):
        row = 1 + (panel_index - 1) // 2
        col = 1 + (panel_index - 1) % 2
        for metric_name in metric_names:
            figure.add_trace(
                go.Scatter(
                    x=steps,
                    y=[metrics[metric_name] for metrics in integrated_history],
                    mode='lines',
                    name=metric_name,
                    legendgroup=metric_name,
                ),
                row=row,
                col=col,
            )
        figure.update_xaxes(title='step', row=row, col=col)
        figure.update_yaxes(type='log', title='value', row=row, col=col)

    figure.update_layout(
        template='plotly_white',
        width=1500,
        height=950,
        title=f'Integrated Training History Through Step {len(integrated_history)}',
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'x': 0.0},
    )
    return figure


def maybe_load_pretrain_cache(runtime: NotebookRuntime) -> bool:
    if not runtime.reuse_pretrain_cache:
        wandb.summary['pretrain_cache_hit'] = False
        return False
    if not runtime.pretrain_cache_path.exists():
        wandb.summary['pretrain_cache_hit'] = False
        return False

    checkpoint = torch.load(runtime.pretrain_cache_path, map_location=runtime.device)
    runtime.encoder.load_state_dict(checkpoint['encoder_state_dict'])
    runtime.decoder.load_state_dict(checkpoint['decoder_state_dict'])
    runtime.critic.load_state_dict(checkpoint['critic_state_dict'])
    runtime.chart_optimizer.load_state_dict(checkpoint['chart_optimizer_state_dict'])
    runtime.critic_optimizer.load_state_dict(checkpoint['critic_optimizer_state_dict'])
    runtime.pretrain_cache_hit = True
    runtime.pretrain_cache_metadata = checkpoint.get('metadata')
    wandb.summary['pretrain_cache_hit'] = True
    if runtime.pretrain_cache_metadata is not None:
        source_run_folder = runtime.pretrain_cache_metadata.get('source_run_folder')
        if source_run_folder is not None:
            wandb.summary['pretrain_cache_source_run_folder'] = source_run_folder
    return True


def save_pretrain_cache(
    runtime: NotebookRuntime,
    *,
    pretrain_metrics: dict[str, float] | None,
    pretrain_monitor_metrics: dict[str, float] | None,
    pretrain_monitor_folder: Path | None,
    critic_metrics: dict[str, float] | None,
    critic_monitor_metrics: dict[str, float] | None,
    critic_monitor_folder: Path | None,
) -> None:
    runtime.pretrain_cache_path.parent.mkdir(parents=True, exist_ok=True)
    metadata = {
        'cache_key': runtime.pretrain_cache_key,
        'source_run_folder': str(runtime.run_folder),
        'created_at': datetime.now().isoformat(),
        'pretrain_metrics': pretrain_metrics,
        'pretrain_monitor_metrics': pretrain_monitor_metrics,
        'pretrain_monitor_folder': None if pretrain_monitor_folder is None else str(pretrain_monitor_folder),
        'critic_metrics': critic_metrics,
        'critic_monitor_metrics': critic_monitor_metrics,
        'critic_monitor_folder': None if critic_monitor_folder is None else str(critic_monitor_folder),
    }
    checkpoint = {
        'cache_config': build_pretrain_cache_config(runtime.config),
        'metadata': metadata,
        'encoder_state_dict': runtime.encoder.state_dict(),
        'decoder_state_dict': runtime.decoder.state_dict(),
        'critic_state_dict': runtime.critic.state_dict(),
        'chart_optimizer_state_dict': runtime.chart_optimizer.state_dict(),
        'critic_optimizer_state_dict': runtime.critic_optimizer.state_dict(),
    }
    torch.save(checkpoint, runtime.pretrain_cache_path)
    runtime.pretrain_cache_metadata = metadata
    runtime.pretrain_cache_path.with_suffix('.json').write_text(json.dumps(metadata, indent=2))
    wandb.summary['pretrain_cache_saved'] = True
    wandb.summary['pretrain_cache_source_run_folder'] = str(runtime.run_folder)


data_batch, data_eta, data_latents = sample_data_latents(runtime, batch_size=4)
prior_latents, generated_samples, model_eta, model_latents = sample_model_latents(runtime, batch_size=4)
{
    'data_batch_shape': tuple(data_batch.shape),
    'data_eta_shape': tuple(data_eta.shape),
    'data_latents_shape': tuple(data_latents.shape),
    'prior_latents_shape': tuple(prior_latents.shape),
    'generated_samples_shape': tuple(generated_samples.shape),
    'model_eta_shape': tuple(model_eta.shape),
    'model_latents_shape': tuple(model_latents.shape),
}

{'data_batch_shape': (4, 128),
 'data_eta_shape': (4, 128),
 'data_latents_shape': (4, 128),
 'prior_latents_shape': (4, 128),
 'generated_samples_shape': (4, 128),
 'model_eta_shape': (4, 128),
 'model_latents_shape': (4, 128)}

## Monitor And Artifact Writers

The monitor writes the gauge-patch artifacts requested in the theory note:
- `data_latents.*`
- `model_latents.*`
- `data_weighted_score_estimate.*`
- `data_weighted_prior_score.*`
- `model_weighted_score_estimate.*`
- `model_weighted_prior_score.*`
- `data_transport_field.*`
- `model_transport_field.*`
- `generated_samples.*`
- `projected_kl.*`
- parquet summaries under `numbers/`


In [7]:
def reconstruction_figure(
    *,
    data_samples: torch.Tensor,
    reconstructions: torch.Tensor,
    reconstruction_error: torch.Tensor,
    labels: torch.Tensor,
) -> go.Figure:
    data_projected = runtime.data_config.project(data_samples.float())
    reconstruction_projected = runtime.data_config.project(reconstructions.float())
    figure = make_subplots(
        rows=1,
        cols=3,
        horizontal_spacing=0.08,
        column_widths=[0.34, 0.34, 0.32],
    )
    labels_cpu = labels.detach().cpu()
    data_projected_cpu = data_projected.detach().cpu()
    reconstruction_projected_cpu = reconstruction_projected.detach().cpu()
    error_cpu = reconstruction_error.detach().cpu()
    for mode_id in range(int(labels_cpu.max().item()) + 1):
        mask = labels_cpu == mode_id
        if not mask.any():
            continue
        color = marker_color(group_id=mode_id)
        figure.add_trace(
            go.Scatter(
                x=data_projected_cpu[mask, 0].tolist(),
                y=data_projected_cpu[mask, 1].tolist(),
                mode='markers',
                name=f'mode {mode_id}',
                legendgroup=f'mode {mode_id}',
                marker={'size': 6, 'opacity': 0.75, 'color': color},
                showlegend=True,
            ),
            row=1,
            col=1,
        )
        figure.add_trace(
            go.Scatter(
                x=reconstruction_projected_cpu[mask, 0].tolist(),
                y=reconstruction_projected_cpu[mask, 1].tolist(),
                mode='markers',
                name=f'mode {mode_id}',
                legendgroup=f'mode {mode_id}',
                marker={'size': 6, 'opacity': 0.75, 'color': color, 'symbol': 'x'},
                showlegend=False,
            ),
            row=1,
            col=2,
        )
        figure.add_trace(
            go.Histogram(
                x=error_cpu[mask].tolist(),
                name=f'mode {mode_id}',
                legendgroup=f'mode {mode_id}',
                marker={'color': color},
                opacity=0.65,
                showlegend=False,
            ),
            row=1,
            col=3,
        )
    figure.update_yaxes(scaleanchor='x', scaleratio=1.0, row=1, col=1)
    figure.update_yaxes(scaleanchor='x2', scaleratio=1.0, row=1, col=2)
    figure.update_layout(
        template='plotly_white',
        width=1500,
        height=520,
        barmode='overlay',
        title='Data reconstruction and reconstruction error',
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'x': 0.0},
    )
    return figure


def latent_scatter_figure(*, latents: torch.Tensor, labels: torch.Tensor, title: str) -> go.Figure:
    projected_latents, off_plane_norm = project_latents_to_pca_space(
        reference_points=latents.float(),
        points=latents.float(),
        projection_dim=2,
        center_at_zero=True,
    )
    figure = go.Figure()
    labels_cpu = labels.detach().cpu()
    projected_cpu = projected_latents.detach().cpu()
    off_plane_cpu = off_plane_norm.detach().cpu()
    for mode_id in range(int(labels_cpu.max().item()) + 1):
        mask = labels_cpu == mode_id
        if not mask.any():
            continue
        figure.add_trace(
            go.Scatter(
                x=projected_cpu[mask, 0].tolist(),
                y=projected_cpu[mask, 1].tolist(),
                mode='markers',
                name=f'mode {mode_id}',
                marker={
                    'size': 6,
                    'opacity': 0.78,
                    'color': marker_color(group_id=mode_id),
                },
                customdata=off_plane_cpu[mask].unsqueeze(-1).tolist(),
                hovertemplate=(
                    f'mode={mode_id}<br>x=%{{x:.3f}}<br>y=%{{y:.3f}}<br>'
                    'off_plane=%{customdata[0]:.4f}<extra></extra>'
                ),
            )
        )
    x_min, x_max, y_min, y_max = latent_square_limits(projected_latents, padding=0.18)
    figure.update_layout(
        template='plotly_white',
        width=850,
        height=850,
        title=title,
        xaxis={'range': [x_min, x_max]},
        yaxis={'range': [y_min, y_max], 'scaleanchor': 'x', 'scaleratio': 1.0},
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'x': 0.0},
    )
    return figure


def vector_field_figure(
    *,
    reference_latents: torch.Tensor,
    labels: torch.Tensor,
    arrow_points: torch.Tensor,
    vectors: torch.Tensor,
    title: str,
    mean_norm_label: str,
) -> go.Figure:
    projected_reference, _ = project_latents_to_pca_space(
        reference_points=reference_latents.float(),
        points=reference_latents.float(),
        projection_dim=2,
        center_at_zero=True,
    )
    projected_arrow_points, _ = project_latents_to_pca_space(
        reference_points=reference_latents.float(),
        points=arrow_points.float(),
        projection_dim=2,
        center_at_zero=True,
    )
    projected_vectors = project_latent_vectors_to_pca_space(
        reference_points=reference_latents.float(),
        vectors=vectors.float(),
        projection_dim=2,
        center_at_zero=True,
    )
    display_length = vector_display_length(projected_arrow_points, fraction=0.06)
    display_vectors = normalize_vectors(vectors=projected_vectors, display_length=display_length)

    figure = go.Figure()
    labels_cpu = labels.detach().cpu()
    projected_reference_cpu = projected_reference.detach().cpu()
    for mode_id in range(int(labels_cpu.max().item()) + 1):
        mask = labels_cpu == mode_id
        if not mask.any():
            continue
        figure.add_trace(
            go.Scatter(
                x=projected_reference_cpu[mask, 0].tolist(),
                y=projected_reference_cpu[mask, 1].tolist(),
                mode='markers',
                name=f'mode {mode_id}',
                marker={
                    'size': 5,
                    'opacity': 0.45,
                    'color': marker_color(group_id=mode_id),
                },
            )
        )
    figure.add_trace(
        go.Scatter(
            x=projected_arrow_points[:, 0].detach().cpu().tolist(),
            y=projected_arrow_points[:, 1].detach().cpu().tolist(),
            mode='markers',
            name='field eval',
            marker={'size': 3, 'opacity': 0.2, 'color': 'black'},
            showlegend=False,
        )
    )
    for index in range(projected_arrow_points.shape[0]):
        x = float(projected_arrow_points[index, 0].cpu())
        y = float(projected_arrow_points[index, 1].cpu())
        u = float(display_vectors[index, 0].cpu())
        v = float(display_vectors[index, 1].cpu())
        figure.add_annotation(
            x=x + u,
            y=y + v,
            ax=x,
            ay=y,
            xref='x',
            yref='y',
            axref='x',
            ayref='y',
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=1.0,
            arrowcolor='rgba(20, 20, 20, 0.65)',
        )
    x_min, x_max, y_min, y_max = latent_square_limits(
        torch.cat([projected_reference, projected_arrow_points], dim=0),
        padding=0.18,
    )
    mean_norm = float(vectors.float().norm(dim=-1).mean().cpu())
    figure.update_layout(
        template='plotly_white',
        width=850,
        height=850,
        title=f"{title} | mean {mean_norm_label}={mean_norm:.3f}",
        xaxis={'range': [x_min, x_max]},
        yaxis={'range': [y_min, y_max], 'scaleanchor': 'x', 'scaleratio': 1.0},
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'x': 0.0},
    )
    return figure


def score_component_figure(
    *,
    reference_latents: torch.Tensor,
    labels: torch.Tensor,
    side_value: int,
    component: str,
    title: str,
) -> go.Figure:
    if component not in {'score_estimate', 'prior_score'}:
        raise ValueError(f'Unknown score component: {component}')
    grid_points, _, _ = build_latent_grid(
        reference_points=reference_latents.float(),
        resolution=runtime.config.monitoring.field_grid_resolution,
        center_at_zero=True,
    )
    with torch.no_grad():
        weighted_score, weighted_prior, _ = estimate_weighted_score_components(
            runtime,
            clean_latents=grid_points.float(),
            side_value=side_value,
        )
    vectors = weighted_score if component == 'score_estimate' else weighted_prior
    return vector_field_figure(
        reference_latents=reference_latents,
        labels=labels,
        arrow_points=grid_points,
        vectors=vectors,
        title=title,
        mean_norm_label='weighted score norm' if component == 'score_estimate' else 'weighted prior norm',
    )


def field_figure(
    *,
    reference_latents: torch.Tensor,
    labels: torch.Tensor,
    side_value: int,
    title: str,
) -> go.Figure:
    grid_points, _, _ = build_latent_grid(
        reference_points=reference_latents.float(),
        resolution=runtime.config.monitoring.field_grid_resolution,
        center_at_zero=True,
    )
    with torch.no_grad():
        _, field, _ = estimate_transport_targets(
            runtime,
            clean_latents=grid_points.float(),
            side_value=side_value,
        )
    return vector_field_figure(
        reference_latents=reference_latents,
        labels=labels,
        arrow_points=grid_points,
        vectors=field,
        title=title,
        mean_norm_label='field norm',
    )


def generated_samples_figure(
    *,
    generated_samples: torch.Tensor,
    data_samples: torch.Tensor,
    data_labels: torch.Tensor,
) -> go.Figure:
    projected_data = runtime.data_config.project(data_samples.float())
    projected_generated = runtime.data_config.project(generated_samples.float())
    figure = go.Figure()
    for mode_id in range(runtime.data_config.num_modes):
        mask = data_labels == mode_id
        figure.add_trace(
            go.Scatter(
                x=projected_data[mask, 0].detach().cpu().tolist(),
                y=projected_data[mask, 1].detach().cpu().tolist(),
                mode='markers',
                name=f'data mode {mode_id}',
                marker={
                    'size': 5,
                    'opacity': 0.3,
                    'color': marker_color(group_id=mode_id),
                },
            )
        )
    figure.add_trace(
        go.Scatter(
            x=projected_generated[:, 0].detach().cpu().tolist(),
            y=projected_generated[:, 1].detach().cpu().tolist(),
            mode='markers',
            name='generated',
            marker={'size': 6, 'opacity': 0.55, 'color': 'rgba(20, 20, 20, 0.85)'},
        )
    )
    x_min, x_max, y_min, y_max = latent_square_limits(
        torch.cat([projected_data, projected_generated], dim=0),
        padding=0.18,
    )
    figure.update_layout(
        template='plotly_white',
        width=850,
        height=850,
        title='Generated samples vs projected data',
        xaxis={'range': [x_min, x_max]},
        yaxis={'range': [y_min, y_max], 'scaleanchor': 'x', 'scaleratio': 1.0},
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'x': 0.0},
    )
    return figure


def projected_kl_figure(*, approximate_kl: torch.Tensor, kde_scales: list[float]) -> go.Figure:
    figure = go.Figure()
    approximate_kl_cpu = approximate_kl.detach().cpu()
    for scale_index, kde_scale in enumerate(kde_scales):
        figure.add_trace(
            go.Bar(
                x=list(range(approximate_kl_cpu.shape[1])),
                y=approximate_kl_cpu[scale_index].tolist(),
                name=f'scale_per_dim={kde_scale:g}',
            )
        )
    figure.update_layout(
        template='plotly_white',
        width=1000,
        height=600,
        title='Projected KL by mode and KDE scale',
        barmode='group',
        xaxis={'title': 'mode_id'},
        yaxis={'title': 'projected KL'},
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'x': 0.0},
    )
    return figure


def write_projected_kl_parquet(
    *,
    step_root: Path,
    approximate_kl: torch.Tensor,
    kde_scales: list[float],
) -> None:
    path = step_root / 'numbers' / 'projected_kl.parquet'
    path.parent.mkdir(parents=True, exist_ok=True)
    approximate_kl_cpu = approximate_kl.detach().cpu().float()
    mode_ids = []
    kde_scale_per_dimension = []
    projected_kl = []
    for scale_index, scale_value in enumerate(kde_scales):
        for mode_id in range(approximate_kl_cpu.shape[1]):
            mode_ids.append(mode_id)
            kde_scale_per_dimension.append(float(scale_value))
            projected_kl.append(float(approximate_kl_cpu[scale_index, mode_id].item()))
    pl.DataFrame(
        {
            'mode_id': mode_ids,
            'kde_scale_per_dimension': kde_scale_per_dimension,
            'projected_kl': projected_kl,
        }
    ).write_parquet(path)


def run_monitor(
    runtime: NotebookRuntime,
    *,
    stage: MonitorStage,
    step: int,
) -> tuple[dict[str, float], Path]:
    runtime.encoder.eval()
    runtime.decoder.eval()
    runtime.critic.eval()

    with torch.no_grad():
        with precision_context(runtime):
            data_samples, data_labels = sample_mode_batch(
                data_config=runtime.data_config,
                device=runtime.device,
                batch_size_per_mode=runtime.config.monitoring.eval_samples_per_mode,
            )
            data_eta = sample_gauge(runtime, data_samples.shape[0])
            data_latents = runtime.encoder(data_samples, data_eta).float()
            reconstructions = runtime.decoder(data_latents).float()
            reconstruction_error = (
                (reconstructions - data_samples)
                .reshape(data_samples.shape[0], -1)
                .norm(dim=-1)
            )

            generated_prior = runtime.prior_config.sample(
                batch_size=runtime.config.monitoring.generated_samples,
            ).to(device=runtime.device, dtype=torch.float32)
            generated_samples = runtime.decoder(generated_prior).float()
            model_eta = sample_gauge(runtime, generated_prior.shape[0])
            model_latents = runtime.encoder(generated_samples, model_eta).float()
            model_labels = assign_mode_ids(runtime, generated_samples)
            approximate_kl = runtime.data_config.approximate_kl(
                data_samples=generated_samples,
                kde_scales=runtime.config.monitoring.kde_scales,
                kl_num_samples=runtime.config.monitoring.kl_num_samples,
                avg_kl_num_batches=runtime.config.monitoring.avg_kl_num_batches,
            )

    folder = step_folder(run_folder=runtime.run_folder, stage=stage, step=step)
    write_figure(
        figure=reconstruction_figure(
            data_samples=data_samples,
            reconstructions=reconstructions,
            reconstruction_error=reconstruction_error,
            labels=data_labels,
        ),
        path_stem=folder / 'data_reconstruction',
    )
    write_figure(
        figure=latent_scatter_figure(
            latents=data_latents,
            labels=data_labels,
            title='Data latents',
        ),
        path_stem=folder / 'data_latents',
    )
    write_figure(
        figure=latent_scatter_figure(
            latents=model_latents,
            labels=model_labels,
            title='Model latents',
        ),
        path_stem=folder / 'model_latents',
    )
    write_figure(
        figure=score_component_figure(
            reference_latents=data_latents,
            labels=data_labels,
            side_value=DATA_SIDE,
            component='score_estimate',
            title='Data weighted score estimate',
        ),
        path_stem=folder / 'data_weighted_score_estimate',
    )
    write_figure(
        figure=score_component_figure(
            reference_latents=data_latents,
            labels=data_labels,
            side_value=DATA_SIDE,
            component='prior_score',
            title='Data weighted prior score',
        ),
        path_stem=folder / 'data_weighted_prior_score',
    )
    write_figure(
        figure=score_component_figure(
            reference_latents=model_latents,
            labels=model_labels,
            side_value=MODEL_SIDE,
            component='score_estimate',
            title='Model weighted score estimate',
        ),
        path_stem=folder / 'model_weighted_score_estimate',
    )
    write_figure(
        figure=score_component_figure(
            reference_latents=model_latents,
            labels=model_labels,
            side_value=MODEL_SIDE,
            component='prior_score',
            title='Model weighted prior score',
        ),
        path_stem=folder / 'model_weighted_prior_score',
    )
    write_figure(
        figure=field_figure(
            reference_latents=data_latents,
            labels=data_labels,
            side_value=DATA_SIDE,
            title='Data transport field',
        ),
        path_stem=folder / 'data_transport_field',
    )
    write_figure(
        figure=field_figure(
            reference_latents=model_latents,
            labels=model_labels,
            side_value=MODEL_SIDE,
            title='Model transport field',
        ),
        path_stem=folder / 'model_transport_field',
    )
    write_figure(
        figure=generated_samples_figure(
            generated_samples=generated_samples,
            data_samples=data_samples,
            data_labels=data_labels,
        ),
        path_stem=folder / 'generated_samples',
    )
    write_figure(
        figure=projected_kl_figure(
            approximate_kl=approximate_kl,
            kde_scales=runtime.config.monitoring.kde_scales,
        ),
        path_stem=folder / 'projected_kl',
    )
    write_mode_value_parquet(
        path=folder / 'numbers' / 'reconstruction_error_norms.parquet',
        mode_ids=data_labels,
        value_column_name='reconstruction_error_norm',
        values=reconstruction_error,
    )
    write_projected_kl_parquet(
        step_root=folder,
        approximate_kl=approximate_kl,
        kde_scales=runtime.config.monitoring.kde_scales,
    )

    metrics = {
        'reconstruction_mean': float(reconstruction_error.mean().cpu()),
        'reconstruction_max': float(reconstruction_error.max().cpu()),
        **{
            f"projected_kl_mean_{str(kde_scale).replace('.', 'p').replace('-', 'm')}": float(
                approximate_kl[scale_index].mean().cpu()
            )
            for scale_index, kde_scale in enumerate(runtime.config.monitoring.kde_scales)
        },
    }
    return metrics, folder


## Chart Pretrain

Pretraining now enforces the full chart repair bundle: stochastic data-fiber reconstruction, canonical data-section reconstruction at `eta = 0`, and canonical model-section decoder/encoder roundtrip constraints at `eta = 0`.
The weak prior anchors remain pretrain-only.


In [8]:
pretrain_history: list[dict[str, float]] = []
pretrain_monitor_metrics: dict[str, float] | None = None
pretrain_monitor_folder: Path | None = None

if maybe_load_pretrain_cache(runtime):
    print(f'[pretrain-cache] loaded {runtime.pretrain_cache_path}')
    if runtime.pretrain_cache_metadata is not None and runtime.pretrain_cache_metadata.get('source_run_folder') is not None:
        print(f"[pretrain-cache] source_run_folder: {runtime.pretrain_cache_metadata['source_run_folder']}")
    print('[pretrain] skipped optimization; using cached warm-start and proceeding to integrated training')
else:
    for step in tqdm(range(1, runtime.config.schedule.pretrain_steps + 1), desc='pretrain'):
        metrics = pretrain_step(runtime)
        pretrain_history.append(metrics)
        log_stage_metrics_to_wandb(stage_name='pretrain', step=step, metrics=metrics)
        if should_log(
            step=step,
            total_steps=runtime.config.schedule.pretrain_steps,
            every=runtime.config.schedule.pretrain_log_every,
        ):
            print(f'[pretrain] step {step}/{runtime.config.schedule.pretrain_steps}: {metrics}')

    if runtime.config.monitoring.run_pretrain_monitor:
        pretrain_monitor_metrics, pretrain_monitor_folder = run_monitor(
            runtime,
            stage=MonitorStage.CHART,
            step=runtime.config.schedule.pretrain_steps,
        )
        log_monitor_snapshot_to_wandb(
            stage_name='pretrain',
            step=runtime.config.schedule.pretrain_steps,
            metrics=pretrain_monitor_metrics,
            folder=pretrain_monitor_folder,
        )

{
    'pretrain_skipped': runtime.pretrain_cache_hit,
    'pretrain_cache_key': runtime.pretrain_cache_key,
    'pretrain_cache_path': str(runtime.pretrain_cache_path),
    'pretrain_cache_source_run_folder': None if runtime.pretrain_cache_metadata is None else runtime.pretrain_cache_metadata.get('source_run_folder'),
    'latest_pretrain_metrics': (None if len(pretrain_history) == 0 else pretrain_history[-1]) if not runtime.pretrain_cache_hit else (None if runtime.pretrain_cache_metadata is None else runtime.pretrain_cache_metadata.get('pretrain_metrics')),
    'pretrain_monitor_metrics': pretrain_monitor_metrics,
    'pretrain_monitor_folder': None if pretrain_monitor_folder is None else str(pretrain_monitor_folder),
}


pretrain:   0%|          | 0/1500 [00:00<?, ?it/s]

[pretrain] step 1/1500: {'pretrain_loss': 1.926289439201355, 'pretrain_repair_loss': 0.46505773067474365, 'data_reconstruction_loss': 0.06754336506128311, 'data_canonical_reconstruction_loss': 0.06225849688053131, 'model_canonical_decoder_loss': 0.15614500641822815, 'model_canonical_encoder_loss': 0.17911086976528168, 'data_anchor_loss': 10.09505844116211, 'model_anchor_loss': 10.941276550292969, 'data_stochastic_l2_loss': 0.15775585174560547, 'model_stochastic_l2_loss': 0.171036034822464, 'data_stochastic_mean_loss': 0.00143968197517097, 'model_stochastic_mean_loss': 0.0017477672081440687, 'data_stochastic_batch_mean_loss': 0.010026748292148113, 'model_stochastic_batch_mean_loss': 0.012128802947700024, 'weak_anchor_loss': 0.0012283313553780317, 'stochastic_prior_loss': 0.02545940689742565, 'data_variance_loss': 0.7267706394195557, 'model_variance_loss': 0.7077733278274536, 'weighted_variance_loss': 1.4345439672470093, 'weighted_model_anchor_loss': 2.1882553100585938, 'model_sample_nor

{'pretrain_skipped': False,
 'pretrain_cache_key': '8c65d38bb6c9b43b',
 'pretrain_cache_path': '/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian/_pretrain_cache/8c65d38bb6c9b43b.pt',
 'pretrain_cache_source_run_folder': None,
 'latest_pretrain_metrics': {'pretrain_loss': 0.03390752896666527,
  'pretrain_repair_loss': 0.006418587639927864,
  'data_reconstruction_loss': 0.0006565657677128911,
  'data_canonical_reconstruction_loss': 0.00013038226461503655,
  'model_canonical_decoder_loss': 0.0047455234453082085,
  'model_canonical_encoder_loss': 0.0008861164096742868,
  'data_anchor_loss': 43.843990325927734,
  'model_anchor_loss': 43.683349609375,
  'data_stochastic_l2_loss': 1.0040205717086792,
  'model_stochastic_l2_loss': 0.9984555244445801,
  'data_stochastic_mean_loss': 0.0002117548428941518,
  'model_stochastic_mean_loss': 0.000245792354689911,
  'data_stochastic_batch_mean_loss': 0.0005387415876612067,
  'model_stochastic_batch_mean_loss': 0.00021166796796

## Side-Conditioned Critic Pretrain

The critic sees the side label explicitly and denoises both stochastic latent pushforwards with the same noise spectrum and the same architecture.


In [9]:
critic_history: list[dict[str, float]] = []
critic_monitor_metrics: dict[str, float] | None = None
critic_monitor_folder: Path | None = None

if runtime.pretrain_cache_hit:
    print(f'[critic] skipped optimization; using cached chart+critic warm-start from {runtime.pretrain_cache_path}')
else:
    for step in tqdm(range(1, runtime.config.schedule.critic_steps + 1), desc='critic_pretrain'):
        metrics = critic_step(runtime)
        critic_history.append(metrics)
        log_stage_metrics_to_wandb(stage_name='critic_pretrain', step=step, metrics=metrics)
        if should_log(
            step=step,
            total_steps=runtime.config.schedule.critic_steps,
            every=runtime.config.schedule.critic_log_every,
        ):
            print(f'[critic] step {step}/{runtime.config.schedule.critic_steps}: {metrics}')

    if runtime.config.monitoring.run_critic_monitor:
        critic_monitor_metrics, critic_monitor_folder = run_monitor(
            runtime,
            stage=MonitorStage.CRITIC,
            step=runtime.config.schedule.critic_steps,
        )
        log_monitor_snapshot_to_wandb(
            stage_name='critic_pretrain',
            step=runtime.config.schedule.critic_steps,
            metrics=critic_monitor_metrics,
            folder=critic_monitor_folder,
        )

    save_pretrain_cache(
        runtime,
        pretrain_metrics=None if len(pretrain_history) == 0 else pretrain_history[-1],
        pretrain_monitor_metrics=pretrain_monitor_metrics,
        pretrain_monitor_folder=pretrain_monitor_folder,
        critic_metrics=None if len(critic_history) == 0 else critic_history[-1],
        critic_monitor_metrics=critic_monitor_metrics,
        critic_monitor_folder=critic_monitor_folder,
    )

{
    'critic_skipped': runtime.pretrain_cache_hit,
    'pretrain_cache_key': runtime.pretrain_cache_key,
    'pretrain_cache_path': str(runtime.pretrain_cache_path),
    'latest_critic_metrics': (None if len(critic_history) == 0 else critic_history[-1]) if not runtime.pretrain_cache_hit else (None if runtime.pretrain_cache_metadata is None else runtime.pretrain_cache_metadata.get('critic_metrics')),
    'critic_monitor_metrics': critic_monitor_metrics,
    'critic_monitor_folder': None if critic_monitor_folder is None else str(critic_monitor_folder),
}


critic_pretrain:   0%|          | 0/2000 [00:00<?, ?it/s]

[critic] step 1/2000: {'critic_loss': 1.1692297458648682, 'critic_data_loss': 1.1711668968200684, 'critic_model_loss': 1.1672924757003784}
[critic] step 100/2000: {'critic_loss': 0.2764725387096405, 'critic_data_loss': 0.2687229514122009, 'critic_model_loss': 0.2842221260070801}
[critic] step 200/2000: {'critic_loss': 0.2592872977256775, 'critic_data_loss': 0.248996764421463, 'critic_model_loss': 0.26957786083221436}
[critic] step 300/2000: {'critic_loss': 0.256389856338501, 'critic_data_loss': 0.24739429354667664, 'critic_model_loss': 0.2653854489326477}
[critic] step 400/2000: {'critic_loss': 0.2499569058418274, 'critic_data_loss': 0.2400907278060913, 'critic_model_loss': 0.2598230838775635}
[critic] step 500/2000: {'critic_loss': 0.24797526001930237, 'critic_data_loss': 0.23512709140777588, 'critic_model_loss': 0.26082342863082886}
[critic] step 600/2000: {'critic_loss': 0.24958321452140808, 'critic_data_loss': 0.23623192310333252, 'critic_model_loss': 0.26293450593948364}
[critic] 

{'critic_skipped': False,
 'pretrain_cache_key': '8c65d38bb6c9b43b',
 'pretrain_cache_path': '/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian/_pretrain_cache/8c65d38bb6c9b43b.pt',
 'latest_critic_metrics': {'critic_loss': 0.24352118372917175,
  'critic_data_loss': 0.23195834457874298,
  'critic_model_loss': 0.2550840377807617},
 'critic_monitor_metrics': {'reconstruction_mean': 0.3995288014411926,
  'reconstruction_max': 1.1934874057769775,
  'projected_kl_mean_0p001': 4.015063285827637,
  'projected_kl_mean_0p003': 3.937699794769287,
  'projected_kl_mean_0p01': 3.928640365600586,
  'projected_kl_mean_0p03': 3.9455981254577637},
 'critic_monitor_folder': '/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian/0401011438-study_ambient128-cdd910/critic_2000'}

## Integrated Chart Repair, Critic Refresh, And Transport

Each integrated step now does three explicit things in order:
1. Repair the stochastic and canonical chart constraints.
2. Refresh the side-conditioned critic for several steps.
3. Transport both the data-side and model-side latent pushforwards toward the symmetry-broken prior, with target construction detached before fitting.


In [ ]:
integrated_history: list[dict[str, float]] = []
integrated_monitor_snapshots: dict[int, dict[str, object]] = {}

for step in tqdm(range(1, runtime.config.schedule.integrated_steps + 1), desc='integrated'):
    repair_metrics = repair_step(runtime)
    critic_metrics = average_metrics(
        [
            critic_step(runtime)
            for _ in range(runtime.config.transport.critic_updates_per_transport_step)
        ]
    )
    transport_metrics = transport_step(runtime)
    combined_metrics = {
        **repair_metrics,
        **critic_metrics,
        **transport_metrics,
    }
    integrated_history.append(combined_metrics)
    log_stage_metrics_to_wandb(stage_name='integrated', step=step, metrics=combined_metrics)

    if should_log(
        step=step,
        total_steps=runtime.config.schedule.integrated_steps,
        every=runtime.config.schedule.integrated_log_every,
    ):
        print(f'[integrated] step {step}/{runtime.config.schedule.integrated_steps}: {combined_metrics}')

    if step in runtime.config.monitoring.integrated_monitor_steps:
        monitor_metrics, monitor_folder = run_monitor(
            runtime,
            stage=MonitorStage.INTEGRATED,
            step=step,
        )
        integrated_monitor_snapshots[step] = {
            'metrics': monitor_metrics,
            'folder': monitor_folder,
        }
        log_monitor_snapshot_to_wandb(
            stage_name='integrated',
            step=step,
            metrics=monitor_metrics,
            folder=monitor_folder,
        )
        print(f'[integrated-monitor] step {step}: {monitor_metrics}')

latest_integrated_metrics = integrated_history[-1]
latest_integrated_metrics


integrated:   0%|          | 0/24000 [00:00<?, ?it/s]

[integrated] step 1/24000: {'repair_loss': 0.00634762505069375, 'repair_data_fiber_reconstruction_loss': 0.0006488980143330991, 'repair_data_canonical_reconstruction_loss': 0.00015327316941693425, 'repair_model_canonical_decoder_loss': 0.004705492407083511, 'repair_model_canonical_encoder_loss': 0.0008399613434448838, 'critic_loss': 0.24379370361566544, 'critic_data_loss': 0.23141222819685936, 'critic_model_loss': 0.2561751678586006, 'transport_loss': 0.15461286902427673, 'data_encoder_transport_loss': 1.9531249563442543e-05, 'data_decoder_transport_loss': 0.0012855078093707561, 'model_encoder_transport_loss': 1.9531249563442543e-05, 'model_decoder_transport_loss': 0.15328830480575562, 'data_field_norm_mean': 19.756610870361328, 'model_field_norm_mean': 19.652183532714844}
[integrated-monitor] step 1: {'reconstruction_mean': 0.39890024065971375, 'reconstruction_max': 0.9996699690818787, 'projected_kl_mean_0p001': 4.114915370941162, 'projected_kl_mean_0p003': 4.000803470611572, 'project

## Integrated History

This plot summarizes the integrated run from step `1` through the current length of `integrated_history`.
Panels are grouped by reconstruction repair, critic losses, transport losses, and transport-field norms.


In [ ]:
if len(integrated_history) == 0:
    raise RuntimeError('integrated_history is empty')

integrated_history_figure = build_integrated_history_figure(integrated_history)
write_figure(
    figure=integrated_history_figure,
    path_stem=runtime.run_folder / 'integrated_history',
)
integrated_history_figure


## Inspect Artifacts

The artifact folders below should now contain the stochastic gauge-patch outputs under `artifacts/multimodal_gaussian/<run_name>`.


In [ ]:
artifact_files = sorted(
    path.relative_to(runtime.run_folder)
    for path in runtime.run_folder.rglob('*')
    if path.is_file()
)
print(f'run_folder: {runtime.run_folder}')
print('recent artifact files:')
for path in artifact_files[-30:]:
    print('  ', path)

if len(integrated_monitor_snapshots) == 0:
    raise RuntimeError('No integrated monitor snapshots were written')
latest_snapshot_step = max(integrated_monitor_snapshots)
latest_snapshot_folder = integrated_monitor_snapshots[latest_snapshot_step]['folder']
latest_projected_kl = pl.read_parquet(latest_snapshot_folder / 'numbers' / 'projected_kl.parquet')
latest_projected_kl.head()

wandb.summary['latest_integrated_monitor_step'] = latest_snapshot_step
wandb.summary['latest_snapshot_folder'] = str(latest_snapshot_folder)
wandb.finish()
